In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/oecd-oda2.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150291 entries, 0 to 150290
Data columns (total 26 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   STRUCTURE           150291 non-null  object 
 1   STRUCTURE_ID        150291 non-null  object 
 2   STRUCTURE_NAME      150291 non-null  object 
 3   ACTION              150291 non-null  object 
 4   DONOR               150291 non-null  object 
 5   Donor               150291 non-null  object 
 6   RECIPIENT           150291 non-null  object 
 7   Recipient           150291 non-null  object 
 8   MEASURE             150291 non-null  int64  
 9   Measure             150291 non-null  object 
 10  UNIT_MEASURE        150291 non-null  object 
 11  Unit of measure     150291 non-null  object 
 12  PRICE_BASE          150291 non-null  object 
 13  Price base          150291 non-null  object 
 14  TIME_PERIOD         150291 non-null  int64  
 15  Time period         0 non-null    

important to know for interpretation: all prices from all years are scaled to match 2023 prices. so inflation does not affect it. 

unit = millions

In [4]:
cdf = df.copy()

cdf['FLOW_TYPE'].unique()

array(['D'], dtype=object)

In [5]:
cdf.drop(columns=["STRUCTURE", "STRUCTURE_ID", "STRUCTURE_NAME", "ACTION", "MEASURE", "Measure", "UNIT_MEASURE", "Unit of measure", "PRICE_BASE", "Price base", "Time period", "Observation value", "BASE_PER", "Base period", "UNIT_MULT", "Unit multiplier", "FLOW_TYPE", "Flow type", "OBS_STATUS", "Observation status"], inplace=True)

In [6]:
cdf.head()

,DONOR,Donor,RECIPIENT,Recipient,TIME_PERIOD,OBS_VALUE
0,PRT,Portugal,F97_X,Middle East unspecified,2009,0.0
1,AUS,Australia,ISR,Israel,1994,0.0
2,AUS,Australia,SVN,Slovenia,1994,0.0
3,AUS,Australia,BIH,Bosnia and Herzegovina,1994,0.0
4,AUS,Australia,HRV,Croatia,1994,0.0


In [7]:
cdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150291 entries, 0 to 150290
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DONOR        150291 non-null  object 
 1   Donor        150291 non-null  object 
 2   RECIPIENT    150291 non-null  object 
 3   Recipient    150291 non-null  object 
 4   TIME_PERIOD  150291 non-null  int64  
 5   OBS_VALUE    150291 non-null  float64
dtypes: float64(1), int64(1), object(4)
memory usage: 6.9+ MB


In [8]:
count_unspecified = cdf['Recipient'].str.contains('unspecified', case=False, na=False).sum()

print("Number of 'unspecified' values:", count_unspecified)

Number of 'unspecified' values: 11348


In [9]:
unspecified_values = cdf.loc[
    cdf['Recipient'].str.contains('unspecified', case=False, na=False),
    'Recipient'
].unique()

print(unspecified_values)

['Middle East unspecified' 'Southern and Central Asia unspecified'
 'South America unspecified' 'Far East Asia unspecified'
 'Central America and the Caribbean unspecified' 'Africa unspecified'
 'Caribbean unspecified' 'States ex-Yugoslavia unspecified'
 'Northern Africa unspecified' 'Europe unspecified'
 'Southern Asia unspecified' 'Asia unspecified' 'America unspecified'
 'Developing countries unspecified' 'Sub-Saharan Africa unspecified'
 'Central Asia unspecified' 'Central America unspecified'
 'Middle Africa unspecified' 'Western Africa unspecified'
 'Eastern Africa unspecified' 'Oceania unspecified'
 'Southern Africa unspecified' 'Melanesia unspecified'
 'Micronesia unspecified' 'Polynesia unspecified']


In [10]:
cdf.drop(cdf[cdf['Recipient'].str.contains('unspecified', case=False, na=False)].index, inplace=True)

cdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 138943 entries, 1 to 150290
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DONOR        138943 non-null  object 
 1   Donor        138943 non-null  object 
 2   RECIPIENT    138943 non-null  object 
 3   Recipient    138943 non-null  object 
 4   TIME_PERIOD  138943 non-null  int64  
 5   OBS_VALUE    138943 non-null  float64
dtypes: float64(1), int64(1), object(4)
memory usage: 7.4+ MB


In [11]:
all_recipients = cdf['Recipient'].unique()

# print all unique values, one per line
for val in all_recipients:
    print(val)

# define the values to remove
abc = ('South America', 'Southern Africa', 'South Africa', 'Asia', 
       'Far East Asia', 'Western Africa', 'Developing countries', 
       'Africa', 'Eastern Africa', 'Sub-Saharan Africa', 
       'Southern and Central Asia', 'Northern America')

# drop rows where Recipient is in abc
cdf = cdf[~cdf['Recipient'].isin(abc)]

Israel
Slovenia
Bosnia and Herzegovina
Croatia
Ecuador
North Macedonia
Northern Mariana Islands
Albania
Armenia
Georgia
Kazakhstan
Kyrgyzstan
Uzbekistan
Samoa
Brazil
Guatemala
Iran
Senegal
Mauritania
Tokelau
Yemen
Chile
Comoros
Chinese Taipei
El Salvador
Democratic Republic of the Congo
Gambia
Costa Rica
Cuba
Jamaica
Central America
Mexico
Eswatini
Jordan
Tajikistan
Turkmenistan
Türkiye
Argentina
Barbados
Bolivia
Lebanon
Sierra Leone
Libya
Niue
Maldives
Venezuela
Mauritius
Micronesia
South Sudan
Cook Islands
Papua New Guinea
Tunisia
Equatorial Guinea
Uruguay
Belize
Guinea-Bissau
Suriname
Côte d’Ivoire
Malaysia
Serbia
Zimbabwe
Thailand
Sri Lanka
Egypt
Viet Nam
Middle East
Vanuatu
Antigua and Barbuda
Saint Kitts and Nevis
Eritrea
Moldova
Bahamas
Tuvalu
Dominican Republic
India
Tanzania
Cabo Verde
Oman
Timor-Leste
Peru
Sudan
Fiji
Mongolia
Namibia
Nigeria
Somalia
Colombia
Chad
Kenya
Europe
Middle Africa
Paraguay
Burundi
Ethiopia
Malawi
Saint Vincent and the Grenadines
Central America and t

In [12]:
# select rows with at least one NaN in any column
nan_rows = cdf[cdf.isna().any(axis=1)]

# choose up to 5 rows
n = min(5, len(nan_rows))

sampled_nan = nan_rows.sample(n=n, random_state=42)
print(sampled_nan)

total_nans = cdf.isna().sum().sum()
print("Total NaNs in DataFrame:", total_nans)

Empty DataFrame
Columns: [DONOR, Donor, RECIPIENT, Recipient, TIME_PERIOD, OBS_VALUE]
Index: []
Total NaNs in DataFrame: 0


In [13]:
cdf.head()

,DONOR,Donor,RECIPIENT,Recipient,TIME_PERIOD,OBS_VALUE
1,AUS,Australia,ISR,Israel,1994,0.0
2,AUS,Australia,SVN,Slovenia,1994,0.0
3,AUS,Australia,BIH,Bosnia and Herzegovina,1994,0.0
4,AUS,Australia,HRV,Croatia,1994,0.0
5,AUS,Australia,ECU,Ecuador,1994,0.0


In [14]:
num_zero_rows = (cdf['OBS_VALUE'] == 0).sum()
print("Number of rows where OBS_VALUE is 0:", num_zero_rows)

cdf = cdf[cdf['OBS_VALUE'] != 0]

Number of rows where OBS_VALUE is 0: 284


In [15]:
import sys
import pandas as pd
import importlib

sys.path.append("../../src")

import country_utils_extended as cs
importlib.reload(cs)

cdf['a_iso3'] = cdf['Donor'].apply(cs.get_iso3)
cdf['b_iso3'] = cdf['Recipient'].apply(cs.get_iso3)

# country_a and country_b being your original columns, a_iso3 b_iso3 will be the new columns added to cdf

cs.check_coverage(cdf, 'a_iso3', 'Donor', 'country a')

cs.check_coverage(cdf, 'b_iso3', 'Recipient', 'country b')


=== country a ===
Total rows with Donor: 125130
Matched to ISO3: 114998 (91.9%)
Unmatched unique values: 3
Unmatched values:
  - DAC countries
  - Non-DAC countries
  - Other countries

=== country b ===
Total rows with Recipient: 125130
Matched to ISO3: 113494 (90.7%)
Unmatched unique values: 12
Unmatched values:
  - America
  - Caribbean
  - Central America
  - Central America and the Caribbean
  - Europe
  - Indus Basin
  - Melanesia
  - Middle Africa
  - Middle East
  - Northern Africa
  - Oceania
  - Polynesia


array(['Central America', 'Middle East', 'Europe', 'Middle Africa',
       'Central America and the Caribbean', 'Polynesia', 'Oceania',
       'Melanesia', 'Caribbean', 'Northern Africa', 'America',
       'Indus Basin'], dtype=object)

In [16]:
# values to drop
to_drop = ['Central America', 'Middle East', 'Europe', 'Middle Africa',
           'Central America and the Caribbean', 'Polynesia', 'Oceania',
           'Melanesia', 'Caribbean', 'Northern Africa', 'America',
           'Indus Basin', 'DAC countries', 'Non-DAC countries', 'Other countries']

# drop rows where Recipient OR Donor is in to_drop
cdf = cdf[~(cdf['Recipient'].isin(to_drop) | cdf['Donor'].isin(to_drop))]

In [17]:
cdf.drop(columns=['DONOR', 'RECIPIENT'], inplace=True)
cdf.rename(columns={'Donor': 'donor', 'Recipient': 'recipient', 'TIME_PERIOD': 'year', 'OBS_VALUE': 'oda_value'}, inplace=True)

In [18]:
cdf.head()

,donor,recipient,year,oda_value,a_iso3,b_iso3
18,Australia,Brazil,1992,0.03,AUS,BRA
19,Australia,Guatemala,1992,0.03,AUS,GTM
20,Australia,Iran,1992,0.03,AUS,IRN
21,Australia,Senegal,1992,0.03,AUS,SEN
22,Australia,Armenia,1993,0.03,AUS,ARM


In [19]:
cdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 104087 entries, 18 to 150290
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   donor      104087 non-null  object 
 1   recipient  104087 non-null  object 
 2   year       104087 non-null  int64  
 3   oda_value  104087 non-null  float64
 4   a_iso3     104087 non-null  object 
 5   b_iso3     104087 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 5.6+ MB


In [20]:
cdf['year'].describe()

count    104087.000000
mean       2009.380816
std           9.355467
min        1992.000000
25%        2002.000000
50%        2010.000000
75%        2018.000000
max        2024.000000
Name: year, dtype: float64

In [25]:
cdf.to_csv("../../data/processed/oecd_dac2_oda.csv", index=False)